In [1]:
!nvidia-smi


Fri Oct  9 05:04:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from numba import cuda
import os

device = cuda.get_current_device()
print(f"Device Name: {device.name.decode('utf-8')}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y},{device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024,64
Warp Size: 32


In [9]:
import numpy as np
import time
from numba import cuda


# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out


# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0


def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # Measure CPU pure loop (skip for N > 1M, too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan')  # Skipped

    # GPU setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Warmup compilation run (same dtype/dims as the real run, so no recompile later)
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU roundtrip measurement (includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU compute-only measurement (data already in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms


if __name__ == "__main__":
    for N in [100,100_0, 10_000, 100_000, 1_000_000,5_000_000, 10_000_000]:
        cpu, total, compute = benchmark_run(N)
        print(f"N={N:>10,} | CPU: {cpu:10.3f} ms | "
              f"GPU total: {total:8.3f} ms | GPU compute: {compute:8.3f} ms")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


N=       100 | CPU:      0.044 ms | GPU total:    0.686 ms | GPU compute:    0.062 ms
N=     1,000 | CPU:      0.242 ms | GPU total:    0.508 ms | GPU compute:    0.058 ms
N=    10,000 | CPU:      2.742 ms | GPU total:    0.553 ms | GPU compute:    0.062 ms
N=   100,000 | CPU:     27.005 ms | GPU total:    0.686 ms | GPU compute:    0.066 ms
N= 1,000,000 | CPU:    275.816 ms | GPU total:    4.541 ms | GPU compute:    0.115 ms
N= 5,000,000 | CPU:        nan ms | GPU total:   13.168 ms | GPU compute:    0.259 ms
N=10,000,000 | CPU:        nan ms | GPU total:   25.406 ms | GPU compute:    0.452 ms


In [13]:
import numpy as np
from numba import cuda

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0  # no bounds check

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")


Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [14]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # 5 blocks = 40 threads

total = blocks_per_grid * threads_per_block
log_b = cuda.device_array(total, dtype=np.int32)
log_t = cuda.device_array(total, dtype=np.int32)
log_g = cuda.device_array(total, dtype=np.int32)
log_a = cuda.device_array(total, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

# copy to host once (indexing device arrays element by element is slow)
b, t, g, a = log_b.copy_to_host(), log_t.copy_to_host(), log_g.copy_to_host(), log_a.copy_to_host()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):
    status = "ACTIVE" if a[i] == 1 else "IDLE (GUARDED)"
    print(f"{g[i]:<10} | {b[i]:<10} | {t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [15]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]          # data race

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
h_ones = np.ones(N, dtype=np.float32)
d_ones = cuda.to_device(h_ones)

tpb = 256
bpg = (N + tpb - 1) // tpb

def run(kernel):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))  # reset to 0 every run
    kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    return d_total.copy_to_host()[0]

# Warm-up so JIT compilation isn't counted
run(naive_sum_kernel)
run(atomic_sum_kernel)

# Five naive trials
print(f"{'Trial':<8}{'Target':>10}{'Naive Output':>16}{'Lost Updates':>16}")
for t in range(1, 6):
    out = run(naive_sum_kernel)
    print(f"Trial {t:<2}{50000.0:>10.1f}{out:>16.1f}{50000.0 - out:>16.1f}")

# Atomic result
atomic_out = run(atomic_sum_kernel)
print(f"\nAtomic sum output: {atomic_out}  (exactly 50000.0? {atomic_out == 50000.0})")

# Timing: average over many runs, data already in VRAM
def time_kernel(kernel, reps=100):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps):
        kernel[bpg, tpb](d_ones, d_total)
    cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / reps

naive_ms = time_kernel(naive_sum_kernel)
atomic_ms = time_kernel(atomic_sum_kernel)
print(f"Naive kernel: {naive_ms:.4f} ms | Atomic kernel: {atomic_ms:.4f} ms")

Trial       Target    Naive Output    Lost Updates
Trial 1    50000.0             2.0         49998.0
Trial 2    50000.0             2.0         49998.0
Trial 3    50000.0             2.0         49998.0
Trial 4    50000.0             2.0         49998.0
Trial 5    50000.0             2.0         49998.0

Atomic sum output: 50000.0  (exactly 50000.0? True)
Naive kernel: 0.0446 ms | Atomic kernel: 0.1823 ms


In [17]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1],
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()

print(f"Generated 2D matrix shape: {result_canvas.shape}. "
      f"Center value: {result_canvas[512, 512]:.3f}")
print(f"Blocks in X: {blocks_2d[0]} | Blocks in Y: {blocks_2d[1]} | "
      f"Total: {blocks_2d[0] * blocks_2d[1]}")
print(f"Center (512, 512): {result_canvas[512, 512]:.6f}")
print(f"Corner (0, 0):     {result_canvas[0, 0]:.6f}")

Generated 2D matrix shape: (1024, 1024). Center value: 1.000
Blocks in X: 64 | Blocks in Y: 64 | Total: 4096
Center (512, 512): 1.000000
Corner (0, 0):     0.000000


In [18]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # GPU name (bytes in some Numba versions, str in others)
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode('utf-8')
    h.update(gpu_name)

    # Top-left 10x10 block of the vignette canvas
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

# REPLACE BOTH VALUES
STUDENT_ID = "YOUR_STUDENT_ID_HERE"   # e.g. "230103456"
CROSSOVER_N = 1000                    # your measured N* from Section 2, as a plain integer

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 091CF5E8D085A3F3


'091CF5E8D085A3F3'